# DFAST Stress Testing — Exploratory Data Analysis

**Dataset**: Fannie Mae Multifamily Loan Performance (Synthetic)

This notebook explores the feature-engineered dataset used for DFAST stress-test modeling.
We examine distributions, default rates, delinquency transitions, and macro-linkage.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.figsize"] = (12, 6)
%matplotlib inline

## 1. Data Loading & Inspection

In [ ]:
DATA_DIR = Path("../data")

acq = pd.read_csv(DATA_DIR / "acquisition.csv", sep="|")
perf = pd.read_csv(DATA_DIR / "performance.csv", sep="|")
macro = pd.read_csv(DATA_DIR / "macro.csv", sep="|")

print(f"Acquisition: {acq.shape}")
print(f"Performance: {perf.shape}")
print(f"Macro:       {macro.shape}")

acq.head()

In [ ]:
print("\n--- Acquisition dtypes ---")
print(acq.dtypes)
print("\n--- Performance dtypes ---")
print(perf.dtypes)

## 2. Missing Value Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Acquisition missing values
acq_missing = acq.isnull().mean().sort_values(ascending=False)
acq_missing[acq_missing > 0].plot(kind="barh", ax=axes[0], color="coral")
axes[0].set_title("Acquisition — Missing Value Rate")
axes[0].set_xlabel("Fraction Missing")

# Performance missing values
perf_missing = perf.isnull().mean().sort_values(ascending=False)
perf_missing[perf_missing > 0].plot(kind="barh", ax=axes[1], color="steelblue")
axes[1].set_title("Performance — Missing Value Rate")
axes[1].set_xlabel("Fraction Missing")

plt.tight_layout()
plt.show()

if acq_missing.sum() == 0 and perf_missing.sum() == 0:
    print("No missing values in either dataset (aside from intentional DSCR nulls).")

## 3. Distribution Plots — Key Numeric Features

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.histplot(acq["original_upb"] / 1e6, bins=50, ax=axes[0, 0], kde=True, color="steelblue")
axes[0, 0].set_title("Original UPB Distribution")
axes[0, 0].set_xlabel("UPB ($M)")

sns.histplot(acq["original_ltv"], bins=40, ax=axes[0, 1], kde=True, color="coral")
axes[0, 1].set_title("Original LTV Distribution")
axes[0, 1].set_xlabel("LTV Ratio")

sns.histplot(acq["original_dscr"].dropna(), bins=40, ax=axes[1, 0], kde=True, color="seagreen")
axes[1, 0].set_title("Original DSCR Distribution")
axes[1, 0].set_xlabel("DSCR")

sns.histplot(acq["note_rate"] * 100, bins=40, ax=axes[1, 1], kde=True, color="goldenrod")
axes[1, 1].set_title("Note Rate Distribution")
axes[1, 1].set_xlabel("Note Rate (%)")

plt.tight_layout()
plt.show()

## 4. Default Rates by Category

In [ ]:
# Create default flag
perf["delinquency_status"] = perf["delinquency_status"].astype(int)
perf["is_default"] = (perf["delinquency_status"] >= 3).astype(int)

# Loan-level default: did the loan ever reach 90+ DPD?
loan_defaults = perf.groupby("loan_id")["is_default"].max().reset_index()
loan_defaults.columns = ["loan_id", "ever_default"]

loan_data = acq.merge(loan_defaults, on="loan_id", how="left")
loan_data["ever_default"] = loan_data["ever_default"].fillna(0).astype(int)

print(f"Overall loan-level default rate: {loan_data['ever_default'].mean():.2%}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# By property type
pt_dr = loan_data.groupby("property_type")["ever_default"].mean().sort_values(ascending=False)
pt_dr.plot(kind="bar", ax=axes[0], color="steelblue")
axes[0].set_title("Default Rate by Property Type")
axes[0].set_ylabel("Default Rate")
axes[0].tick_params(axis='x', rotation=45)

# By state (top 10)
REGION_MAP = {
    'CT': 'NE', 'ME': 'NE', 'MA': 'NE', 'NH': 'NE', 'NJ': 'NE', 'NY': 'NE', 'PA': 'NE', 'RI': 'NE', 'VT': 'NE', 'DE': 'NE', 'MD': 'NE', 'DC': 'NE',
    'AL': 'SE', 'AR': 'SE', 'FL': 'SE', 'GA': 'SE', 'KY': 'SE', 'LA': 'SE', 'MS': 'SE', 'NC': 'SE', 'SC': 'SE', 'TN': 'SE', 'VA': 'SE', 'WV': 'SE',
    'IL': 'MW', 'IN': 'MW', 'IA': 'MW', 'KS': 'MW', 'MI': 'MW', 'MN': 'MW', 'MO': 'MW', 'NE': 'MW', 'ND': 'MW', 'OH': 'MW', 'SD': 'MW', 'WI': 'MW',
    'AK': 'W', 'AZ': 'W', 'CA': 'W', 'CO': 'W', 'HI': 'W', 'ID': 'W', 'MT': 'W', 'NV': 'W', 'NM': 'W', 'OK': 'W', 'OR': 'W', 'TX': 'W', 'UT': 'W', 'WA': 'W', 'WY': 'W',
}
loan_data["region"] = loan_data["state"].map(REGION_MAP).fillna("Other")
region_dr = loan_data.groupby("region")["ever_default"].mean().sort_values(ascending=False)
region_dr.plot(kind="bar", ax=axes[1], color="coral")
axes[1].set_title("Default Rate by Region")
axes[1].set_ylabel("Default Rate")
axes[1].tick_params(axis='x', rotation=0)

# By origination vintage
loan_data["orig_year"] = pd.to_datetime(loan_data["origination_date"]).dt.year
vintage_dr = loan_data.groupby("orig_year")["ever_default"].mean()
vintage_dr.plot(kind="bar", ax=axes[2], color="seagreen")
axes[2].set_title("Default Rate by Origination Vintage")
axes[2].set_ylabel("Default Rate")
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 5. Delinquency Transition Matrix

In [ ]:
perf_sorted = perf.sort_values(["loan_id", "reporting_period"])
perf_sorted["next_dq"] = perf_sorted.groupby("loan_id")["delinquency_status"].shift(-1)
transitions = perf_sorted.dropna(subset=["next_dq"])

trans_matrix = pd.crosstab(
    transitions["delinquency_status"],
    transitions["next_dq"].astype(int),
    normalize="index"
)
trans_matrix.index.name = "From"
trans_matrix.columns.name = "To"

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    trans_matrix, annot=True, fmt=".2%", cmap="YlOrRd",
    ax=ax, cbar_kws={"label": "Transition Probability"}
)
ax.set_title("Monthly Delinquency Transition Matrix")
ax.set_xlabel("Next Month State")
ax.set_ylabel("Current Month State")
plt.tight_layout()
plt.show()

print("\nTransition Matrix (row = from, col = to):")
print(trans_matrix.round(4))

## 6. Correlation Heatmap

In [ ]:
numeric_cols = ["original_upb", "original_ltv", "original_dscr", "note_rate", "number_of_units"]
corr = acq[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r", center=0, ax=ax)
ax.set_title("Correlation Matrix — Loan Characteristics")
plt.tight_layout()
plt.show()

## 7. Default Rate Over Time

In [ ]:
perf["reporting_period"] = pd.to_datetime(perf["reporting_period"])
monthly_dr = perf.groupby(perf["reporting_period"].dt.to_period("M"))["is_default"].mean()

fig, ax = plt.subplots(figsize=(14, 5))
monthly_dr.plot(ax=ax, color="crimson", linewidth=2)
ax.fill_between(range(len(monthly_dr)), monthly_dr.values, alpha=0.15, color="crimson")
ax.set_title("Monthly Default Rate (90+ DPD) Over Time")
ax.set_ylabel("Default Rate")
ax.set_xlabel("Reporting Month")
plt.tight_layout()
plt.show()

## 8. Summary Statistics

In [ ]:
summary = acq[["original_upb", "original_ltv", "original_dscr", "note_rate", "number_of_units"]].describe().T
summary["missing_%"] = acq[["original_upb", "original_ltv", "original_dscr", "note_rate", "number_of_units"]].isnull().mean() * 100

print("\n=== Acquisition Summary Statistics ===")
print(summary.round(4).to_string())

print(f"\n=== Portfolio Overview ===")
print(f"Total loans:           {len(acq):,}")
print(f"Total UPB:             ${acq['original_upb'].sum():,.0f}")
print(f"Performance records:   {len(perf):,}")
print(f"Loan-level default rate: {loan_data['ever_default'].mean():.2%}")
print(f"Obs-level default rate:  {perf['is_default'].mean():.2%}")